In [0]:
import os, time, json, requests
from datetime import date, timedelta

API_KEY = dbutils.secrets.get("heatcheck", "datagov_api_key")
HEADERS = {"x-api-key": API_KEY}
URL = "https://api-open.data.gov.sg/v2/real-time/api/weather"
FOLDER = "/Volumes/workspace/heatcheck/raw/wbgt"

def get_with_retry(params):
    for attempt in range(5):
        r = requests.get(URL, params=params, headers=HEADERS, timeout=30)
        if r.status_code == 429:          # too many requests: wait and retry
            time.sleep(10 * (attempt + 1))
            continue
        r.raise_for_status()
        return r
    raise RuntimeError(f"Gave up after 5 tries: {params}")

def backfill_day(d):
    if os.path.exists(f"{FOLDER}/wbgt_backfill_{d:%Y%m%d}_p1.json"):
        return "already done"
    params = {"api": "wbgt", "date": d.isoformat()}
    pages = []
    while True:
        r = get_with_retry(params)
        pages.append(r.text)
        token = r.json().get("data", {}).get("paginationToken")
        if not token:
            break
        params["paginationToken"] = token
        time.sleep(0.5)
    for i, text in enumerate(pages, start=1):   # save only once the whole day is complete
        with open(f"{FOLDER}/wbgt_backfill_{d:%Y%m%d}_p{i}.json", "w") as f:
            f.write(text)
    records = sum(len(json.loads(t).get("data", {}).get("records", [])) for t in pages)
    return f"{len(pages)} page(s), {records} readings"

In [0]:
for d in [date(2025, 3, 1), date(2025, 3, 2), date(2025, 3, 3)]:
    print(d, backfill_day(d))

In [0]:
d, end = date(2025, 2, 1), date.today() - timedelta(days=1)
while d <= end:
    result = backfill_day(d)
    if d.day == 1 or "done" not in result:
        print(d, result)
    d += timedelta(days=1)
    time.sleep(0.5)
print("Backfill finished")